# Brain Tumour Classification with Explainable AI

**Models:** ResNet50 and EfficientNetB0 with Grad-CAM and SHAP

**Dataset:** Kaggle Brain Tumor MRI Dataset (Nickparvar, 2021)
- Training: 5,712 images (pre-split)
- Testing: 1,311 images (pre-split)
- Classes: 4 (Glioma, Meningioma, No Tumour, Pituitary)

---

## 1. Install Dependencies

In [ ]:
!pip install kagglehub timm torchvision scikit-learn matplotlib seaborn tqdm scipy h5py -q

## 2. Imports

In [ ]:
import os
import time
import random
import warnings
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from PIL import Image
import timm
import kagglehub

from sklearn.metrics import (
    confusion_matrix, classification_report, accuracy_score,
    precision_recall_fscore_support, roc_curve, auc,
    f1_score, precision_score, recall_score
)
from sklearn.preprocessing import label_binarize
from collections import Counter
from tqdm.auto import tqdm

import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

print('All imports completed successfully.')

## 3. Random Seeds and Device Configuration

In [ ]:
def set_seed(seed=42):
    """Set random seeds for reproducibility across all libraries."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU:    {torch.cuda.get_device_name(0)}')

## 4. Download Dataset

In [ ]:
print('Downloading Brain Tumor MRI Dataset...')
dataset_path = kagglehub.dataset_download('masoudnickparvar/brain-tumor-mri-dataset')
print(f'Dataset downloaded to: {dataset_path}')

TRAIN_DIR = os.path.join(dataset_path, 'Training')
TEST_DIR = os.path.join(dataset_path, 'Testing')

print(f'\nTraining directory: {TRAIN_DIR}')
print(f'Testing directory:  {TEST_DIR}')
print(f'\nTraining directory exists: {os.path.exists(TRAIN_DIR)}')
print(f'Testing directory exists:  {os.path.exists(TEST_DIR)}')

## 5. Data Preparation

The Kaggle dataset includes a pre-defined Training/Testing split, so no
additional conversion or splitting is required. This cell verifies the
directory structure and reports class-level statistics.

In [ ]:
print('=' * 60)
print('DATASET STATISTICS')
print('=' * 60)

classes = sorted(os.listdir(TRAIN_DIR))
print(f'Classes found: {classes}')

total_train = 0
total_test = 0
for cls in classes:
    train_count = len(os.listdir(os.path.join(TRAIN_DIR, cls)))
    test_count = len(os.listdir(os.path.join(TEST_DIR, cls)))
    total_train += train_count
    total_test += test_count
    print(f'{cls:15s}: {train_count:4d} train, {test_count:4d} test')
print('=' * 60)
print(f'{"TOTAL":15s}: {total_train:4d} train, {total_test:4d} test')
print('=' * 60)

---

## 6. Configuration

In [ ]:
NUM_CLASSES = 4
BATCH_SIZE = 32
EPOCHS = 30
EARLY_STOP_PATIENCE = 5       # Stop if no improvement for 5 consecutive epochs
LEARNING_RATE_RESNET = 1e-4
LEARNING_RATE_EFFNET = 5e-5
DATASET_NAME = 'Kaggle'
CLASS_NAMES = ['glioma', 'meningioma', 'notumor', 'pituitary']


print('=' * 60)
print('CONFIGURATION')
print('=' * 60)
print(f'Dataset:            {DATASET_NAME}')
print(f'Classes:            {NUM_CLASSES} - {CLASS_NAMES}')
print(f'Batch size:         {BATCH_SIZE}')
print(f'Max epochs:         {EPOCHS}')
print(f'Early stop after:   {EARLY_STOP_PATIENCE} epochs with no improvement')
print(f'ResNet50 LR:        {LEARNING_RATE_RESNET}')
print(f'EffNetB0 LR:        {LEARNING_RATE_EFFNET}')
print(f'Device:             {DEVICE}')
print('=' * 60)

---

## 7. Exploratory Data Analysis

In [ ]:
# 7.1 Class Distribution

def plot_class_distribution(train_dir, test_dir, class_names):
    """Plot class distribution for training and testing sets."""
    train_counts = []
    test_counts = []

    for cls in class_names:
        train_path = os.path.join(train_dir, cls)
        test_path = os.path.join(test_dir, cls)
        train_counts.append(len(os.listdir(train_path)))
        test_counts.append(len(os.listdir(test_path)))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].bar(class_names, train_counts, color='steelblue', alpha=0.8)
    axes[0].set_title('Training Set Distribution', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Class')
    axes[0].set_ylabel('Number of Images')
    axes[0].grid(axis='y', alpha=0.3)
    for i, count in enumerate(train_counts):
        axes[0].text(i, count + 20, str(count), ha='center', fontweight='bold')

    axes[1].bar(class_names, test_counts, color='coral', alpha=0.8)
    axes[1].set_title('Testing Set Distribution', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Class')
    axes[1].set_ylabel('Number of Images')
    axes[1].grid(axis='y', alpha=0.3)
    for i, count in enumerate(test_counts):
        axes[1].text(i, count + 5, str(count), ha='center', fontweight='bold')

    plt.tight_layout()
    plt.savefig('kaggle_class_distribution.png', dpi=300, bbox_inches='tight')
    plt.show()

    print('\n' + '=' * 60)
    print('CLASS DISTRIBUTION STATISTICS')
    print('=' * 60)
    print(f'{"Class":<15} {"Training":<12} {"Testing":<12} {"Total":<10}')
    print('-' * 60)
    total_train = 0
    total_test = 0
    for i, cls in enumerate(class_names):
        total_train += train_counts[i]
        total_test += test_counts[i]
        print(f'{cls:<15} {train_counts[i]:<12} {test_counts[i]:<12} '
              f'{train_counts[i] + test_counts[i]:<10}')
    print('-' * 60)
    print(f'{"TOTAL":<15} {total_train:<12} {total_test:<12} '
          f'{total_train + total_test:<10}')
    print('=' * 60)

plot_class_distribution(TRAIN_DIR, TEST_DIR, CLASS_NAMES)

In [ ]:
# 7.2 Sample Images from Each Class

def display_sample_images(train_dir, class_names, samples_per_class=4):
    """Display representative sample images from each class."""
    fig, axes = plt.subplots(len(class_names), samples_per_class, figsize=(15, 12))
    fig.suptitle('Sample Images from Each Class', fontsize=16, fontweight='bold', y=0.995)

    for i, cls in enumerate(class_names):
        class_path = os.path.join(train_dir, cls)
        images = os.listdir(class_path)[:samples_per_class]

        for j, img_name in enumerate(images):
            img_path = os.path.join(class_path, img_name)
            img = Image.open(img_path)
            axes[i, j].imshow(img)
            axes[i, j].axis('off')
            if j == 0:
                axes[i, j].set_title(f'{cls.upper()}',
                                      fontsize=12, fontweight='bold', loc='left')

    plt.tight_layout()
    plt.savefig('kaggle_sample_images.png', dpi=300, bbox_inches='tight')
    plt.show()

display_sample_images(TRAIN_DIR, CLASS_NAMES, samples_per_class=4)

In [ ]:
# 7.3 Image Properties Analysis (Dimensions, Channels)

def analyze_image_properties(train_dir, class_names, num_samples=100):
    """Analyse image dimensions and colour mode properties."""
    widths = []
    heights = []
    modes = []

    for cls in class_names:
        class_path = os.path.join(train_dir, cls)
        images = os.listdir(class_path)
        sample_images = random.sample(images, min(num_samples // len(class_names), len(images)))

        for img_name in sample_images:
            img_path = os.path.join(class_path, img_name)
            with Image.open(img_path) as img:
                widths.append(img.width)
                heights.append(img.height)
                modes.append(img.mode)

    print('\n' + '=' * 60)
    print('IMAGE PROPERTIES ANALYSIS')
    print('=' * 60)
    print(f'Samples analysed: {len(widths)}')
    print(f'\nImage Dimensions:')
    print(f'  Width  -- Min: {min(widths)}, Max: {max(widths)}, Mean: {np.mean(widths):.1f}')
    print(f'  Height -- Min: {min(heights)}, Max: {max(heights)}, Mean: {np.mean(heights):.1f}')
    print(f'\nColour Modes: {set(modes)}')
    print(f'  Most common: {max(set(modes), key=modes.count)}')
    print('=' * 60)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].hist(widths, bins=20, color='skyblue', alpha=0.7, edgecolor='black')
    axes[0].set_title('Image Width Distribution', fontweight='bold')
    axes[0].set_xlabel('Width (pixels)')
    axes[0].set_ylabel('Frequency')
    axes[0].grid(alpha=0.3)

    axes[1].hist(heights, bins=20, color='lightcoral', alpha=0.7, edgecolor='black')
    axes[1].set_title('Image Height Distribution', fontweight='bold')
    axes[1].set_xlabel('Height (pixels)')
    axes[1].set_ylabel('Frequency')
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig('kaggle_image_properties.png', dpi=300, bbox_inches='tight')
    plt.show()

analyze_image_properties(TRAIN_DIR, CLASS_NAMES, num_samples=100)

---

## 8. Data Augmentation Visualisation

In [ ]:
def visualize_augmentation(train_dir, class_names):
    """Demonstrate the effect of augmentation transforms on a sample image."""
    sample_class = class_names[0]
    class_path = os.path.join(train_dir, sample_class)
    sample_image = os.listdir(class_path)[0]
    img_path = os.path.join(class_path, sample_image)
    img = Image.open(img_path)

    augmentation_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomRotation(15),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
        transforms.ColorJitter(brightness=0.1, contrast=0.1)
    ])

    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    fig.suptitle('Data Augmentation Examples', fontsize=16, fontweight='bold')

    axes[0, 0].imshow(img)
    axes[0, 0].set_title('Original', fontweight='bold')
    axes[0, 0].axis('off')

    for i in range(1, 10):
        row = i // 5
        col = i % 5
        augmented = augmentation_transforms(img)
        axes[row, col].imshow(augmented)
        axes[row, col].set_title(f'Augmented {i}', fontweight='bold')
        axes[row, col].axis('off')

    plt.tight_layout()
    plt.savefig('kaggle_augmentation_effects.png', dpi=300, bbox_inches='tight')
    plt.show()

    print('\nAugmentation Techniques Applied:')
    print('  Random Rotation: +/-15 degrees')
    print('  Random Horizontal Flip: 50% probability')
    print('  Random Translation: +/-10%')
    print('  Colour Jitter: Brightness +/-10%, Contrast +/-10%')

visualize_augmentation(TRAIN_DIR, CLASS_NAMES)

## 9. Dataset Class Definition

In [ ]:
class BrainTumorDataset(Dataset):
    """Brain Tumor MRI Dataset wrapper around ImageFolder."""

    def __init__(self, data_dir, transform=None):
        self.data = ImageFolder(data_dir, transform=transform)
        self.classes = self.data.classes
        self.class_to_idx = self.data.class_to_idx

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

    def get_class_distribution(self):
        """Return the distribution of classes in the dataset."""
        return Counter(self.data.targets)

print('BrainTumorDataset class defined.')

## 10. Model Architecture

In [ ]:
class BrainTumorClassifier(nn.Module):
    """Classification head on top of a pre-trained feature extractor."""

    def __init__(self, base_model, num_classes=4):
        super(BrainTumorClassifier, self).__init__()
        self.base_model = base_model
        num_features = base_model.num_features

        self.classifier = nn.Sequential(
            nn.Linear(num_features, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        features = self.base_model(x)
        return self.classifier(features)

print('BrainTumorClassifier defined.')

## 11. Base Model Loader with Model-Specific Preprocessing

Uses `timm.data.resolve_data_config()` to retrieve the correct normalisation
parameters for each architecture.

In [ ]:
def get_base_model(model_name):
    """Load a pre-trained model and configure architecture-specific transforms."""
    print(f'\n{"=" * 60}')
    print(f'Loading {model_name.upper()}')
    print('=' * 60)

    base_model = timm.create_model(model_name, pretrained=True, num_classes=0)
    base_model_cfg = timm.data.resolve_data_config(model=base_model)

    print(f'\nModel Configuration:')
    print(f'  Input size:    {base_model_cfg["input_size"]}')
    print(f'  Mean:          {base_model_cfg["mean"]}')
    print(f'  Std:           {base_model_cfg["std"]}')
    print(f'  Num features:  {base_model.num_features}')

    standard_transforms = transforms.Compose([
        transforms.Resize(
            (base_model_cfg['input_size'][1], base_model_cfg['input_size'][2]),
            interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.ToTensor(),
        transforms.Normalize(mean=base_model_cfg['mean'], std=base_model_cfg['std'])
    ])

    augmented_transforms = transforms.Compose([
        transforms.Resize(
            (base_model_cfg['input_size'][1], base_model_cfg['input_size'][2]),
            interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.RandomRotation(15),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
        transforms.ColorJitter(brightness=0.1, contrast=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=base_model_cfg['mean'], std=base_model_cfg['std'])
    ])

    print('  Transforms configured.')
    print('=' * 60)
    return base_model, standard_transforms, augmented_transforms

print('Base model loader defined.')

## 12. DataLoader Creator

In [ ]:
def create_dataloader(train_folder, test_folder, standard_transforms,
                      augmented_transforms, batch_size=32):
    """Create training and testing DataLoaders with class-balanced weights."""

    temp_dataset = ImageFolder(train_folder)
    class_counts = Counter(temp_dataset.targets)

    print('\n' + '=' * 60)
    print('CLASS DISTRIBUTION IN TRAINING SET')
    print('=' * 60)
    for class_idx, count in sorted(class_counts.items()):
        class_name = temp_dataset.classes[class_idx]
        percentage = (count / len(temp_dataset)) * 100
        print(f'{class_name:15s}: {count:4d} images ({percentage:5.2f}%)')
    print('=' * 60)

    total_samples = len(temp_dataset)
    class_weights = [total_samples / (len(class_counts) * class_counts[i])
                     for i in range(len(class_counts))]
    class_weights = torch.FloatTensor(class_weights)

    print('\nClass Weights (for balancing):')
    for class_idx, weight in enumerate(class_weights):
        class_name = temp_dataset.classes[class_idx]
        print(f'  {class_name:15s}: {weight:.4f}')
    print('=' * 60)

    train_dataset = BrainTumorDataset(train_folder, transform=augmented_transforms)
    test_dataset = BrainTumorDataset(test_folder, transform=standard_transforms)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                              num_workers=0, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False,
                             num_workers=0, pin_memory=True)

    print(f'\nDataLoaders created:')
    print(f'  Training batches: {len(train_loader)}')
    print(f'  Testing batches:  {len(test_loader)}')
    print(f'  Batch size:       {batch_size}')

    return train_loader, test_loader, class_weights

print('Data loading function defined.')

## 13. Training Function

In [ ]:
def train_model(model, train_loader, test_loader, class_weights, device='cpu',
                epochs=30, lr=1e-4, model_name='model',
                patience=5):
    """Train the model with early stopping based on test accuracy."""

    model = model.to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-7)

    history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}

    best_test_acc = 0.0
    best_epoch = 0
    epochs_no_improve = 0
    best_state = None

    print(f'\n{"=" * 60}')
    print(f'STARTING TRAINING: {model_name}')
    print(f'{"=" * 60}')
    print(f'Device:            {device}')
    print(f'Learning rate:     {lr}')
    print(f'Max epochs:        {epochs}')
    print(f'Early stopping:    {patience} epochs patience')
    print(f'Training samples:  {len(train_loader.dataset)}')
    print(f'Testing samples:   {len(test_loader.dataset)}')
    print('=' * 60)

    for epoch in range(epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0

        train_pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}',
                          leave=False)
        for inputs, labels in train_pbar:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()
            train_pbar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'acc': f'{100 * train_correct / train_total:.2f}%'
            })

        train_loss = train_loss / len(train_loader)
        train_acc = 100 * train_correct / train_total

        # Evaluation phase
        model.eval()
        test_loss = 0.0
        test_correct = 0
        test_total = 0

        with torch.no_grad():
            for inputs, labels in test_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                test_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                test_total += labels.size(0)
                test_correct += (predicted == labels).sum().item()

        test_loss = test_loss / len(test_loader)
        test_acc = 100 * test_correct / test_total

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['test_loss'].append(test_loss)
        history['test_acc'].append(test_acc)

        old_lr = optimizer.param_groups[0]['lr']
        scheduler.step(test_loss)
        current_lr = optimizer.param_groups[0]['lr']

        # Early stopping check
        improved = ''
        if test_acc > best_test_acc:
            best_test_acc = test_acc
            best_epoch = epoch + 1
            epochs_no_improve = 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            improved = '  New best '
        else:
            epochs_no_improve += 1

        print(f'\nEpoch [{epoch + 1}/{epochs}]')
        print(f'  Train Loss: {train_loss:.4f}  Acc: {train_acc:.2f}%')
        print(f'  Test Loss:  {test_loss:.4f}  Acc: {test_acc:.2f}%{improved}')
        lr_msg = f'  Learning Rate: {current_lr:.2e}'
        if current_lr < old_lr:
            lr_msg += f' (reduced from {old_lr:.2e})'
        print(lr_msg)
        print(f'  Early Stop Counter: {epochs_no_improve}/{patience}')

        if epochs_no_improve >= patience:
            print(f'\n{"=" * 60}')
            print(f'EARLY STOPPING TRIGGERED at epoch {epoch + 1}')
            print(f'No improvement in test accuracy for {patience} '
                  f'consecutive epochs.')
            print(f'{"=" * 60}')
            break

    # Restore best model
    if best_state is not None:
        model.load_state_dict(best_state)

    actual_epochs = len(history['train_loss'])
    print(f'\n{"=" * 60}')
    print(f'Training complete: {model_name}')
    print(f'  Epochs run:       {actual_epochs}/{epochs}')
    print(f'  Best test acc:    {best_test_acc:.2f}% (epoch {best_epoch})')
    if actual_epochs < epochs:
        print(f'  Early stopped:    Yes (saved {epochs - actual_epochs} epochs)')
    else:
        print(f'  Early stopped:    No (ran all {epochs} epochs)')
    print('=' * 60)

    return model, history

print('Training function defined with early stopping.')

## 14. Evaluation Function

In [ ]:
def evaluate_model(model, test_loader, device='cpu'):
    """Comprehensive model evaluation returning per-class and overall metrics."""
    model = model.to(device)
    model.eval()

    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for inputs, labels in tqdm(test_loader, desc='Evaluating'):
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            probs = torch.softmax(outputs, dim=1)
            _, predicted = torch.max(outputs, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    preds = np.array(all_preds)
    labels = np.array(all_labels)
    probs = np.array(all_probs)

    accuracy = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average=None, zero_division=0)
    overall_precision, overall_recall, overall_f1, _ = precision_recall_fscore_support(
        labels, preds, average='macro', zero_division=0)

    cm = confusion_matrix(labels, preds)

    results = {
        'accuracy': accuracy,
        'confusion_matrix': cm,
        'per_class_precision': precision,
        'per_class_recall': recall,
        'per_class_f1': f1,
        'overall_precision': overall_precision,
        'overall_recall': overall_recall,
        'overall_f1': overall_f1,
        'all_probs': probs,
        'all_labels': labels,
        'all_preds': preds
    }

    return results

print('Evaluation function defined.')

## 15. Plotting Functions

In [ ]:
def plot_confusion_matrix(results, model_name):
    """Generate and display a confusion matrix heatmap."""
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(results['confusion_matrix'], annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(f'{model_name}: Confusion Matrix ({DATASET_NAME})')
    plt.tight_layout()
    return fig


def plot_roc_curves(results, model_name):
    """Generate and display per-class ROC curves."""
    fig, ax = plt.subplots(figsize=(8, 6))
    y_bin = label_binarize(results['all_labels'], classes=range(NUM_CLASSES))
    colours = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3']
    for i, (cls, col) in enumerate(zip(CLASS_NAMES, colours[:NUM_CLASSES])):
        fpr, tpr, _ = roc_curve(y_bin[:, i], results['all_probs'][:, i])
        ax.plot(fpr, tpr, color=col, lw=2,
                label=f'{cls} (AUC = {auc(fpr, tpr):.4f})')
    ax.plot([0, 1], [0, 1], 'k--', lw=1)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'{model_name}: ROC Curves ({DATASET_NAME})')
    ax.legend(loc='lower right')
    plt.tight_layout()
    return fig


def plot_history(history, model_name):
    """Generate and display training history plots (loss and accuracy)."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ep = range(1, len(history['train_loss']) + 1)

    ax1.plot(ep, history['train_loss'], 'b-', marker='o', markersize=4, label='Train')
    ax1.plot(ep, history['test_loss'], 'r-', marker='s', markersize=4, label='Test')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title(f'{model_name}: Loss ({DATASET_NAME})')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(ep, history['train_acc'], 'b-', marker='o', markersize=4, label='Train')
    ax2.plot(ep, history['test_acc'], 'r-', marker='s', markersize=4, label='Test')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy (%)')
    ax2.set_title(f'{model_name}: Accuracy ({DATASET_NAME})')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    return fig


def print_results(results, model_name, train_time):
    """Print formatted evaluation results."""
    print(f'\n{"=" * 60}')
    print(f'{model_name}: FINAL RESULTS ({DATASET_NAME})')
    print(f'{"=" * 60}')
    print(f'Accuracy:       {results["accuracy"] * 100:.2f}%')
    print(f'Precision:      {results["overall_precision"] * 100:.2f}%')
    print(f'Recall:         {results["overall_recall"] * 100:.2f}%')
    print(f'F1-Score:       {results["overall_f1"] * 100:.2f}%')
    print(f'Training Time:  {train_time:.2f} min')
    print(f'\nPer-Class Metrics:')
    print(f'{"-" * 60}')
    for i, c in enumerate(CLASS_NAMES):
        print(f'  {c:15s}  P={results["per_class_precision"][i] * 100:5.2f}%  '
              f'R={results["per_class_recall"][i] * 100:5.2f}%  '
              f'F1={results["per_class_f1"][i] * 100:5.2f}%')
    print(f'{"=" * 60}')

print('Plotting functions defined.')

---

## 16. Train ResNet50

In [ ]:
print('RESNET50 TRAINING')

resnet_base, standard_transforms_resnet, augmented_transforms_resnet = get_base_model('resnet50')

train_loader_resnet, test_loader_resnet, class_weights = create_dataloader(
    TRAIN_DIR, TEST_DIR,
    standard_transforms_resnet, augmented_transforms_resnet,
    batch_size=BATCH_SIZE)

resnet_model = BrainTumorClassifier(resnet_base, num_classes=NUM_CLASSES)
total_params = sum(p.numel() for p in resnet_model.parameters())
trainable_params = sum(p.numel() for p in resnet_model.parameters() if p.requires_grad)
print(f'\nModel Parameters:')
print(f'  Total:     {total_params:,}')
print(f'  Trainable: {trainable_params:,}')

t0 = time.time()
resnet_trained, history_resnet = train_model(
    resnet_model, train_loader_resnet, test_loader_resnet, class_weights,
    device=DEVICE, epochs=EPOCHS, lr=LEARNING_RATE_RESNET, model_name='resnet50')
time_resnet = (time.time() - t0) / 60

results_resnet = evaluate_model(resnet_trained, test_loader_resnet, device=DEVICE)
print_results(results_resnet, 'ResNet50', time_resnet)

fig = plot_confusion_matrix(results_resnet, 'ResNet50')
fig.savefig('kaggle_resnet50_cm.png', dpi=300, bbox_inches='tight')
plt.show()

fig = plot_roc_curves(results_resnet, 'ResNet50')
fig.savefig('kaggle_resnet50_roc.png', dpi=300, bbox_inches='tight')
plt.show()

fig = plot_history(history_resnet, 'ResNet50')
fig.savefig('kaggle_resnet50_history.png', dpi=300, bbox_inches='tight')
plt.show()

---

## 17. Train EfficientNetB0

In [ ]:
print('EFFICIENTNETB0 TRAINING')

effnet_base, standard_transforms_effnet, augmented_transforms_effnet = get_base_model('efficientnet_b0')

train_loader_effnet, test_loader_effnet, _ = create_dataloader(
    TRAIN_DIR, TEST_DIR,
    standard_transforms_effnet, augmented_transforms_effnet,
    batch_size=BATCH_SIZE)

effnet_model = BrainTumorClassifier(effnet_base, num_classes=NUM_CLASSES)
total_params = sum(p.numel() for p in effnet_model.parameters())
trainable_params = sum(p.numel() for p in effnet_model.parameters() if p.requires_grad)
print(f'\nModel Parameters:')
print(f'  Total:     {total_params:,}')
print(f'  Trainable: {trainable_params:,}')

t0 = time.time()
effnet_trained, history_effnet = train_model(
    effnet_model, train_loader_effnet, test_loader_effnet, class_weights,
    device=DEVICE, epochs=EPOCHS, lr=LEARNING_RATE_EFFNET, model_name='efficientnetb0')
time_effnet = (time.time() - t0) / 60

results_effnet = evaluate_model(effnet_trained, test_loader_effnet, device=DEVICE)
print_results(results_effnet, 'EfficientNetB0', time_effnet)

fig = plot_confusion_matrix(results_effnet, 'EfficientNetB0')
fig.savefig('kaggle_effnetb0_cm.png', dpi=300, bbox_inches='tight')
plt.show()

fig = plot_roc_curves(results_effnet, 'EfficientNetB0')
fig.savefig('kaggle_effnetb0_roc.png', dpi=300, bbox_inches='tight')
plt.show()

fig = plot_history(history_effnet, 'EfficientNetB0')
fig.savefig('kaggle_effnetb0_history.png', dpi=300, bbox_inches='tight')
plt.show()

---

## 18. Classification Reports

In [ ]:
print('ResNet50 Classification Report:')
print(classification_report(results_resnet['all_labels'], results_resnet['all_preds'],
                            target_names=CLASS_NAMES, digits=4))

print('\nEfficientNetB0 Classification Report:')
print(classification_report(results_effnet['all_labels'], results_effnet['all_preds'],
                            target_names=CLASS_NAMES, digits=4))

---

## 19. Model Comparison Table

Side-by-side quantitative comparison of both architectures on the Kaggle dataset.

In [ ]:
import pandas as pd

# 19.1 Tabular Comparison

print('=' * 70)
print('KAGGLE: MODEL COMPARISON TABLE')
print('=' * 70)

metrics = {
    'Accuracy (%)':       [f'{results_resnet["accuracy"]*100:.2f}', f'{results_effnet["accuracy"]*100:.2f}'],
    'Precision (%)':      [f'{results_resnet["overall_precision"]*100:.2f}', f'{results_effnet["overall_precision"]*100:.2f}'],
    'Recall (%)':         [f'{results_resnet["overall_recall"]*100:.2f}', f'{results_effnet["overall_recall"]*100:.2f}'],
    'F1-Score (%)':       [f'{results_resnet["overall_f1"]*100:.2f}', f'{results_effnet["overall_f1"]*100:.2f}'],
    'Training Time (min)':[f'{time_resnet:.2f}', f'{time_effnet:.2f}'],
    'Parameters':         ['23.8M', '5.5M'],
}

print(f'\n{"Metric":<22s} {"ResNet50":>14s} {"EfficientNetB0":>16s} {"Winner":>12s}')
print('-' * 66)

for metric, values in metrics.items():
    v1, v2 = values
    try:
        f1, f2 = float(v1.replace('M', '')), float(v2.replace('M', ''))
        if 'Time' in metric or 'Parameters' in metric:
            winner = 'ResNet50' if f1 < f2 else 'EfficientNetB0'
        else:
            winner = 'ResNet50' if f1 > f2 else 'EfficientNetB0'
    except ValueError:
        winner = '-'
    print(f'{metric:<22s} {v1:>14s} {v2:>16s} {winner:>12s}')

# 19.2 Per-Class Performance Comparison

print(f'\n\n{"=" * 70}')
print('PER-CLASS PERFORMANCE COMPARISON')
print(f'{"=" * 70}')
print(f'\n{"Class":<15s} {"ResNet50":>32s} {"EfficientNetB0":>32s}')
print(f'{"":<15s} {"P":>8s} {"R":>8s} {"F1":>8s} {"   P":>8s} {"R":>8s} {"F1":>8s}')
print('-' * 79)

for i, cls in enumerate(CLASS_NAMES):
    rp = results_resnet['per_class_precision'][i] * 100
    rr = results_resnet['per_class_recall'][i] * 100
    rf = results_resnet['per_class_f1'][i] * 100
    ep = results_effnet['per_class_precision'][i] * 100
    er = results_effnet['per_class_recall'][i] * 100
    ef = results_effnet['per_class_f1'][i] * 100
    print(f'{cls:<15s} {rp:>7.2f}% {rr:>7.2f}% {rf:>7.2f}% {ep:>7.2f}% {er:>7.2f}% {ef:>7.2f}%')

# 19.3 Visual Comparison Charts

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

metric_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
resnet_vals = [results_resnet['accuracy'] * 100, results_resnet['overall_precision'] * 100,
               results_resnet['overall_recall'] * 100, results_resnet['overall_f1'] * 100]
effnet_vals = [results_effnet['accuracy'] * 100, results_effnet['overall_precision'] * 100,
               results_effnet['overall_recall'] * 100, results_effnet['overall_f1'] * 100]

x = np.arange(len(metric_names))
width = 0.35
bars1 = axes[0].bar(x - width / 2, resnet_vals, width, label='ResNet50',
                     color='#3498db', edgecolor='black')
bars2 = axes[0].bar(x + width / 2, effnet_vals, width, label='EfficientNetB0',
                     color='#e74c3c', edgecolor='black')
axes[0].set_ylabel('Score (%)')
axes[0].set_title(f'Overall Performance ({DATASET_NAME})')
axes[0].set_xticks(x)
axes[0].set_xticklabels(metric_names)
axes[0].legend()
axes[0].set_ylim(0, 105)
for bar in bars1:
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                 f'{bar.get_height():.1f}', ha='center', fontsize=9)
for bar in bars2:
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                 f'{bar.get_height():.1f}', ha='center', fontsize=9)

x = np.arange(len(CLASS_NAMES))
bars1 = axes[1].bar(x - width / 2, results_resnet['per_class_f1'] * 100, width,
                     label='ResNet50', color='#3498db', edgecolor='black')
bars2 = axes[1].bar(x + width / 2, results_effnet['per_class_f1'] * 100, width,
                     label='EfficientNetB0', color='#e74c3c', edgecolor='black')
axes[1].set_ylabel('F1-Score (%)')
axes[1].set_title(f'Per-Class F1-Score ({DATASET_NAME})')
axes[1].set_xticks(x)
axes[1].set_xticklabels(CLASS_NAMES, rotation=15)
axes[1].legend()
axes[1].set_ylim(0, 105)
for bar in bars1:
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                 f'{bar.get_height():.1f}', ha='center', fontsize=9)
for bar in bars2:
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                 f'{bar.get_height():.1f}', ha='center', fontsize=9)

plt.suptitle(f'Kaggle Dataset: Model Comparison ({EPOCHS} Epochs)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('kaggle_model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

comparison_df = pd.DataFrame({
    'Metric': ['Accuracy (%)', 'Precision (%)', 'Recall (%)', 'F1-Score (%)'],
    'ResNet50': [f'{results_resnet["accuracy"]*100:.2f}', f'{results_resnet["overall_precision"]*100:.2f}',
                 f'{results_resnet["overall_recall"]*100:.2f}', f'{results_resnet["overall_f1"]*100:.2f}'],
    'EfficientNetB0': [f'{results_effnet["accuracy"]*100:.2f}', f'{results_effnet["overall_precision"]*100:.2f}',
                       f'{results_effnet["overall_recall"]*100:.2f}', f'{results_effnet["overall_f1"]*100:.2f}']
})
comparison_df.to_csv('kaggle_model_comparison.csv', index=False)
print('\nComparison saved: kaggle_model_comparison.csv')

## 20. Save Models

In [ ]:
torch.save(resnet_trained.state_dict(), 'resnet50_kaggle_final.pth')
torch.save(effnet_trained.state_dict(), 'efficientnetb0_kaggle_final.pth')

print('=' * 60)
print('MODELS SAVED')
print('=' * 60)
print('  resnet50_kaggle_final.pth')
print('  efficientnetb0_kaggle_final.pth')
print('=' * 60)

---

## 21. Explainable AI: Grad-CAM Analysis

Grad-CAM (Gradient-weighted Class Activation Mapping) generates visual attention
maps highlighting the regions of MRI scans most influential to each model's
classification decisions.

In [ ]:
# 21.1 Install and Import Grad-CAM Dependencies

!pip install grad-cam opencv-python -q

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
import cv2

print('Grad-CAM libraries installed and imported.')

In [ ]:
# 21.2 Grad-CAM Visualisation Function

def apply_gradcam(model, input_tensor, target_layer, class_idx=None):
    """Apply Grad-CAM and return the heatmap and predicted class."""
    model.eval()
    with torch.no_grad():
        output = model(input_tensor.to(DEVICE))
        predicted_class = output.argmax(dim=1).item()
    if class_idx is None:
        class_idx = predicted_class
    cam = GradCAM(model=model, target_layers=[target_layer])
    targets = [ClassifierOutputTarget(class_idx)]
    grayscale_cam = cam(input_tensor=input_tensor.to(DEVICE), targets=targets)
    grayscale_cam = grayscale_cam[0, :]
    return grayscale_cam, predicted_class


def get_denormalized_image(img_tensor, mean, std):
    """Convert a normalised tensor back to a displayable [0, 1] numpy array."""
    img = img_tensor.cpu().numpy().transpose(1, 2, 0)
    img = img * np.array(std) + np.array(mean)
    img = np.clip(img, 0, 1)
    return img


def visualize_gradcam_grid(model, dataset, target_layer, class_names,
                            mean, std, num_samples=2, model_name='Model'):
    """Generate a grid of Grad-CAM visualisations: Original | Heatmap | Overlay."""
    samples_per_class = {i: [] for i in range(len(class_names))}
    for idx in range(len(dataset)):
        img, label = dataset[idx]
        if len(samples_per_class[label]) < num_samples:
            samples_per_class[label].append((img, label))
        if all(len(v) >= num_samples for v in samples_per_class.values()):
            break

    n_classes = len(class_names)
    total_rows = n_classes * num_samples
    fig, axes = plt.subplots(total_rows, 3, figsize=(15, 5 * total_rows))
    if total_rows == 1:
        axes = axes.reshape(1, -1)

    row = 0
    for cls_idx in range(n_classes):
        for img_tensor, label in samples_per_class[cls_idx]:
            img_display = get_denormalized_image(img_tensor, mean, std)
            input_batch = img_tensor.unsqueeze(0)
            cam_map, pred_class = apply_gradcam(model, input_batch, target_layer)

            overlay = show_cam_on_image(img_display, cam_map, use_rgb=True)
            heatmap_coloured = cv2.applyColorMap(np.uint8(255 * cam_map), cv2.COLORMAP_JET)
            heatmap_coloured = cv2.cvtColor(heatmap_coloured, cv2.COLOR_BGR2RGB)

            correct_str = 'Correct' if pred_class == label else 'Incorrect'
            title_colour = 'green' if pred_class == label else 'red'

            axes[row, 0].imshow(img_display)
            axes[row, 0].set_title(f'True: {class_names[label]}', fontsize=11)
            axes[row, 0].axis('off')

            axes[row, 1].imshow(heatmap_coloured)
            axes[row, 1].set_title('Grad-CAM Heatmap', fontsize=11)
            axes[row, 1].axis('off')

            axes[row, 2].imshow(overlay)
            axes[row, 2].set_title(f'Pred: {class_names[pred_class]} ({correct_str})',
                                    fontsize=11, color=title_colour)
            axes[row, 2].axis('off')
            row += 1

    plt.suptitle(f'{model_name}: Grad-CAM Visualisation ({DATASET_NAME})',
                 fontsize=16, fontweight='bold')
    plt.tight_layout()
    save_path = f'kaggle_{model_name.lower().replace(" ", "_")}_gradcam.png'
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f'Figure saved: {save_path}')

print('Grad-CAM functions defined.')

In [ ]:
# 21.3 Grad-CAM: ResNet50

print('\n' + '=' * 60)
print('GRAD-CAM ANALYSIS: RESNET50 (Kaggle)')
print('=' * 60)

resnet_target_layer = resnet_trained.base_model.layer4[-1]
print(f'Target layer: {resnet_target_layer.__class__.__name__}')

resnet_mean = [0.485, 0.456, 0.406]
resnet_std = [0.229, 0.224, 0.225]

test_dataset_resnet = BrainTumorDataset(TEST_DIR, transform=standard_transforms_resnet)
print(f'Test dataset size: {len(test_dataset_resnet)}')
print('Generating Grad-CAM visualisations...\n')

visualize_gradcam_grid(
    model=resnet_trained,
    dataset=test_dataset_resnet,
    target_layer=resnet_target_layer,
    class_names=CLASS_NAMES,
    mean=resnet_mean,
    std=resnet_std,
    num_samples=2,
    model_name='ResNet50'
)

In [ ]:
# 21.4 Grad-CAM: EfficientNetB0

print('\n' + '=' * 60)
print('GRAD-CAM ANALYSIS: EFFICIENTNETB0 (Kaggle)')
print('=' * 60)

effnet_target_layer = effnet_trained.base_model.conv_head
print(f'Target layer: {effnet_target_layer.__class__.__name__}')

effnet_mean = [0.485, 0.456, 0.406]
effnet_std = [0.229, 0.224, 0.225]

test_dataset_effnet = BrainTumorDataset(TEST_DIR, transform=standard_transforms_effnet)
print(f'Test dataset size: {len(test_dataset_effnet)}')
print('Generating Grad-CAM visualisations...\n')

visualize_gradcam_grid(
    model=effnet_trained,
    dataset=test_dataset_effnet,
    target_layer=effnet_target_layer,
    class_names=CLASS_NAMES,
    mean=effnet_mean,
    std=effnet_std,
    num_samples=2,
    model_name='EfficientNetB0'
)

---

## 22. Explainable AI: SHAP Analysis

SHAP (SHapley Additive exPlanations) provides pixel-level feature importance
analysis using the PartitionExplainer, a model-agnostic approach that avoids
gradient-related in-place operation errors.

In [ ]:
# 22.1 Install and Import SHAP Dependencies

!pip install shap -q

import shap
import gc

print(f'SHAP version: {shap.__version__}')

In [ ]:
# 22.2 SHAP Analysis Function (PartitionExplainer)

def visualize_shap_analysis(model, dataset, class_names, device, mean, std,
                             num_samples=3, model_name='Model'):
    """Perform SHAP analysis using PartitionExplainer and generate visualisations."""
    print(f'\n{"=" * 60}')
    print(f'SHAP ANALYSIS: {model_name.upper()} ({DATASET_NAME})')
    print(f'{"=" * 60}')

    model.eval()

    samples = []
    class_tracker = {i: 0 for i in range(len(class_names))}
    max_per_class = max(1, num_samples // len(class_names))

    for idx in range(len(dataset)):
        img, label = dataset[idx]
        if class_tracker[label] < max_per_class:
            samples.append((img, label))
            class_tracker[label] += 1
        if len(samples) >= num_samples:
            break

    if not samples:
        print('ERROR: No samples found in dataset.')
        return False

    images_shap = []
    labels_list = []
    mean_arr = np.array(mean)
    std_arr = np.array(std)
    for img_tensor, label in samples:
        img_np = get_denormalized_image(img_tensor, mean, std)
        img_uint8 = (img_np * 255).astype(np.uint8)
        images_shap.append(img_uint8)
        labels_list.append(label)

    images_array = np.array(images_shap)

    def predict_fn(images):
        preds = []
        for img in images:
            img_float = img.astype(np.float32) / 255.0
            img_norm = (img_float - mean_arr) / std_arr
            img_chw = np.transpose(img_norm, (2, 0, 1))
            img_tensor = torch.from_numpy(img_chw).float().unsqueeze(0).to(device)
            with torch.no_grad():
                output = model(img_tensor)
                probs = torch.nn.functional.softmax(output, dim=1)
                preds.append(probs.cpu().numpy()[0])
        return np.array(preds)

    print(f'Creating PartitionExplainer for {len(images_array)} images...')
    print(f'Estimated time: {len(images_array) * 3}-{len(images_array) * 5} minutes')

    masker = shap.maskers.Image("inpaint_telea", images_array[0].shape)
    explainer = shap.Explainer(predict_fn, masker, output_names=class_names)

    print('Computing SHAP values...')
    shap_values = explainer(images_array, max_evals=100, batch_size=10)

    print('Generating SHAP visualisations...')
    shap.image_plot(shap_values, pixel_values=images_array)

    fig, axes = plt.subplots(len(images_array), len(class_names) + 1,
                              figsize=(4 * (len(class_names) + 1), 4 * len(images_array)))
    if len(images_array) == 1:
        axes = axes.reshape(1, -1)

    for i in range(len(images_array)):
        axes[i, 0].imshow(images_array[i])
        axes[i, 0].set_title(f'True: {class_names[labels_list[i]]}', fontsize=10)
        axes[i, 0].axis('off')

        for j, cls in enumerate(class_names):
            sv = shap_values[i].values[..., j]
            sv_grey = np.sum(np.abs(sv), axis=-1) if sv.ndim == 3 else np.abs(sv)
            axes[i, j + 1].imshow(images_array[i], alpha=0.3)
            axes[i, j + 1].imshow(sv_grey, cmap='hot', alpha=0.7)
            axes[i, j + 1].set_title(f'SHAP: {cls}', fontsize=10)
            axes[i, j + 1].axis('off')

    plt.suptitle(f'{model_name}: SHAP Explanations ({DATASET_NAME})',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    save_path = f'kaggle_{model_name.lower().replace(" ", "_")}_shap.png'
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f'Figure saved: {save_path}')
    return True

print('SHAP functions defined.')

In [ ]:
# 22.3 SHAP: ResNet50

resnet_mean = [0.485, 0.456, 0.406]
resnet_std = [0.229, 0.224, 0.225]

test_dataset_shap_r = BrainTumorDataset(TEST_DIR, transform=standard_transforms_resnet)

shap_success_resnet = visualize_shap_analysis(
    model=resnet_trained,
    dataset=test_dataset_shap_r,
    class_names=CLASS_NAMES,
    device=DEVICE,
    mean=resnet_mean,
    std=resnet_std,
    num_samples=4,
    model_name='ResNet50'
)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('GPU memory released.')

In [ ]:
# 22.4 Clear GPU Memory

import gc
torch.cuda.empty_cache()
gc.collect()
print('GPU memory released.')

In [ ]:
# 22.5 SHAP: EfficientNetB0

effnet_mean = [0.485, 0.456, 0.406]
effnet_std = [0.229, 0.224, 0.225]

test_dataset_shap_e = BrainTumorDataset(TEST_DIR, transform=standard_transforms_effnet)

shap_success_effnet = visualize_shap_analysis(
    model=effnet_trained,
    dataset=test_dataset_shap_e,
    class_names=CLASS_NAMES,
    device=DEVICE,
    mean=effnet_mean,
    std=effnet_std,
    num_samples=4,
    model_name='EfficientNetB0'
)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('GPU memory released.')

---

## 23. Detection System with Explainability

Complete detection pipeline that combines classification with Grad-CAM
visualisation for clinical interpretability.

In [ ]:
# 23.1 Detection Function

import torch.nn.functional as F

def detect_with_xai(img_path, model, target_layer, class_names, device,
                     model_name='Model', image_size=224):
    """Complete detection with Grad-CAM explainability visualisation."""
    try:
        img = Image.open(img_path).convert('RGB')
        img_display = img.copy()

        img_resized = img.resize((image_size, image_size))
        img_array = np.array(img_resized).astype(np.float32) / 255.0

        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])

        img_normalized = (img_array - mean) / std
        img_chw = np.transpose(img_normalized, (2, 0, 1))
        img_tensor = torch.from_numpy(img_chw).float().unsqueeze(0).to(device)

        model.eval()
        with torch.no_grad():
            output = model(img_tensor)
            probabilities = F.softmax(output, dim=1)[0]
            predicted_class_index = output.argmax(dim=1).item()
            confidence_score = probabilities[predicted_class_index].item()

        cam = GradCAM(model=model, target_layers=[target_layer])
        targets = [ClassifierOutputTarget(predicted_class_index)]
        grayscale_cam = cam(input_tensor=img_tensor, targets=targets)[0, :]

        cam_overlay = show_cam_on_image(img_array, grayscale_cam, use_rgb=True,
                                         image_weight=0.6)

        if class_names[predicted_class_index] == 'notumor':
            result = 'No Tumour Detected'
            result_color = 'green'
        else:
            result = f'Tumour Detected: {class_names[predicted_class_index].capitalize()}'
            result_color = 'red'

        fig = plt.figure(figsize=(20, 5))

        plt.subplot(1, 5, 1)
        plt.imshow(img_display)
        plt.title('Original MRI', fontsize=12, fontweight='bold')
        plt.axis('off')

        plt.subplot(1, 5, 2)
        plt.imshow(img_array)
        plt.title('Preprocessed', fontsize=12, fontweight='bold')
        plt.axis('off')

        plt.subplot(1, 5, 3)
        plt.imshow(grayscale_cam, cmap='jet')
        plt.title('Grad-CAM Heatmap', fontsize=12, fontweight='bold')
        plt.axis('off')

        plt.subplot(1, 5, 4)
        plt.imshow(cam_overlay)
        plt.title(f'{result}\nConfidence: {confidence_score * 100:.2f}%',
                  fontsize=12, fontweight='bold', color=result_color)
        plt.axis('off')

        plt.subplot(1, 5, 5)
        colors = ['green' if cls == 'notumor' else 'red' for cls in class_names]
        bars = plt.barh(class_names,
                         [probabilities[i].item() for i in range(len(class_names))],
                         color=colors, alpha=0.6)
        plt.title('Class Probabilities', fontsize=12, fontweight='bold')
        plt.xlabel('Probability')
        plt.xlim([0, 1])
        plt.grid(axis='x', alpha=0.3)
        bars[predicted_class_index].set_alpha(1.0)
        bars[predicted_class_index].set_edgecolor('black')
        bars[predicted_class_index].set_linewidth(2)

        fig.suptitle(f'{model_name} Detection: {result}',
                     fontsize=16, fontweight='bold', color=result_color, y=1.02)
        plt.tight_layout()
        plt.show()

        print(f'\n{"=" * 70}')
        print(f'{model_name.upper()} DETECTION RESULTS')
        print(f'{"=" * 70}')
        print(f'Image:      {os.path.basename(img_path)}')
        print(f'Result:     {result}')
        print(f'Confidence: {confidence_score * 100:.2f}%')
        print(f'\nAll Class Probabilities:')
        for i, label in enumerate(class_names):
            marker = '-->' if i == predicted_class_index else '   '
            print(f'  {marker} {label.capitalize():12s}: {probabilities[i] * 100:6.2f}%')
        print(f'{"=" * 70}')

        return predicted_class_index, confidence_score, grayscale_cam

    except Exception as e:
        print(f'Error: {str(e)}')
        return None, None, None

print('Detection function defined.')

In [ ]:
# 23.2 Test Detection System: ResNet50

print('\n' + '#' * 70)
print('DETECTION SYSTEM WITH XAI: TESTING')


print('\nCollecting test samples (one per class)...')

sample_images = {}
class_counts = {i: 0 for i in range(len(CLASS_NAMES))}

for images, labels in test_loader_resnet:
    for i in range(images.size(0)):
        label = labels[i].item()
        class_name = CLASS_NAMES[label]
        if class_name not in sample_images and class_counts[label] < 1:
            img_denorm = images[i].cpu().numpy().transpose(1, 2, 0)
            img_denorm = img_denorm * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
            img_denorm = np.clip(img_denorm, 0, 1)
            temp_path = f'/tmp/test_{class_name}.png'
            Image.fromarray((img_denorm * 255).astype(np.uint8)).save(temp_path)
            sample_images[class_name] = (temp_path, images[i], label)
            class_counts[label] += 1
        if len(sample_images) == len(CLASS_NAMES):
            break
    if len(sample_images) == len(CLASS_NAMES):
        break

print(f'Collected {len(sample_images)} samples.\n')

print('=' * 70)
print('RESNET50 DETECTION WITH XAI')
print('=' * 70)

resnet_target_layer = resnet_trained.base_model.layer4[-1]

for class_name, (img_path, img_tensor, true_label) in sample_images.items():
    print(f'\n{"*" * 70}')
    print(f'Testing: {class_name.upper()}')
    print(f'{"*" * 70}')
    detect_with_xai(
        img_path=img_path, model=resnet_trained,
        target_layer=resnet_target_layer, class_names=CLASS_NAMES,
        device=DEVICE, model_name='ResNet50')

print('\n' + '=' * 70)
print('ResNet50 detection with XAI complete.')
print('=' * 70)

In [ ]:
# 23.3 Test Detection System: EfficientNetB0

print('\n' + '=' * 70)
print('EFFICIENTNETB0 DETECTION WITH XAI')
print('=' * 70)

effnet_target_layer = effnet_trained.base_model.conv_head

sample_images_effnet = {}
class_counts = {i: 0 for i in range(len(CLASS_NAMES))}

for images, labels in test_loader_effnet:
    for i in range(images.size(0)):
        label = labels[i].item()
        class_name = CLASS_NAMES[label]
        if class_name not in sample_images_effnet and class_counts[label] < 1:
            img_denorm = images[i].cpu().numpy().transpose(1, 2, 0)
            img_denorm = img_denorm * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
            img_denorm = np.clip(img_denorm, 0, 1)
            temp_path = f'/tmp/test_effnet_{class_name}.png'
            Image.fromarray((img_denorm * 255).astype(np.uint8)).save(temp_path)
            sample_images_effnet[class_name] = (temp_path, images[i], label)
            class_counts[label] += 1
        if len(sample_images_effnet) == len(CLASS_NAMES):
            break
    if len(sample_images_effnet) == len(CLASS_NAMES):
        break

for class_name, (img_path, img_tensor, true_label) in sample_images_effnet.items():
    print(f'\n{"*" * 70}')
    print(f'Testing: {class_name.upper()}')
    print(f'{"*" * 70}')
    detect_with_xai(
        img_path=img_path, model=effnet_trained,
        target_layer=effnet_target_layer, class_names=CLASS_NAMES,
        device=DEVICE, model_name='EfficientNetB0')

print('\n' + '=' * 70)
print('EfficientNetB0 detection with XAI complete.')
print('=' * 70)

---

## 24. Experiment Complete

In [ ]:
print('ALL EXPERIMENTS COMPLETE')

print(f'\nDataset:   {DATASET_NAME}')
print(f'Classes:   {NUM_CLASSES} ({", ".join(CLASS_NAMES)})')
print(f'Epochs:    {EPOCHS}')
print(f'Batch:     {BATCH_SIZE}')

print(f'\n{"MODEL PERFORMANCE SUMMARY":^70s}')
print(f'\n{"Metric":<22s} {"ResNet50":>14s} {"EfficientNetB0":>16s}')
print('-' * 54)
print(f'{"Accuracy (%)":<22s} {results_resnet["accuracy"]*100:>13.2f}% {results_effnet["accuracy"]*100:>15.2f}%')
print(f'{"Precision (%)":<22s} {results_resnet["overall_precision"]*100:>13.2f}% {results_effnet["overall_precision"]*100:>15.2f}%')
print(f'{"Recall (%)":<22s} {results_resnet["overall_recall"]*100:>13.2f}% {results_effnet["overall_recall"]*100:>15.2f}%')
print(f'{"F1-Score (%)":<22s} {results_resnet["overall_f1"]*100:>13.2f}% {results_effnet["overall_f1"]*100:>15.2f}%')
print(f'{"Training Time (min)":<22s} {time_resnet:>13.2f}  {time_effnet:>15.2f} ')

print(f'\n{"EXPLAINABILITY ANALYSIS STATUS":^70s}')
print(f'  Grad-CAM (ResNet50):       Complete')
print(f'  Grad-CAM (EfficientNetB0): Complete')
try:
    print(f'  SHAP (ResNet50):           {"Complete" if shap_success_resnet else "Partial"}')
    print(f'  SHAP (EfficientNetB0):     {"Complete" if shap_success_effnet else "Partial"}')
except NameError:
    print(f'  SHAP:                      Check individual cells above')

print(f'\nALL KAGGLE EXPERIMENTS COMPLETE')

##  Interactive Grad-CAM

In [ ]:
# Interactive Grad-CAM: Select an image and compare both models

import ipywidgets as widgets
from IPython.display import display, clear_output

# Collect all test images organised by class
test_image_paths = {}
for cls in sorted(os.listdir(TEST_DIR)):
    cls_path = os.path.join(TEST_DIR, cls)
    if os.path.isdir(cls_path):
        imgs = sorted([f for f in os.listdir(cls_path)
                       if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        test_image_paths[cls] = imgs

# Widgets
class_dropdown = widgets.Dropdown(
    options=list(test_image_paths.keys()),
    description='Class:',
    style={'description_width': '60px'}
)

image_dropdown = widgets.Dropdown(
    options=test_image_paths[class_dropdown.value][:20],
    description='Image:',
    style={'description_width': '60px'}
)

opacity_slider = widgets.FloatSlider(
    value=0.6, min=0.0, max=1.0, step=0.05,
    description='Overlay:',
    style={'description_width': '60px'}
)

output_area = widgets.Output()

def update_images(*args):
    image_dropdown.options = test_image_paths[class_dropdown.value][:20]

class_dropdown.observe(update_images, names='value')


def compute_attention_stats(heatmap, threshold=0.5):
    """Compute statistics about where the model focuses its attention."""
    h, w = heatmap.shape
    total_pixels = h * w

    # Percentage of image the model considers important
    active_pixels = np.sum(heatmap >= threshold)
    focus_pct = (active_pixels / total_pixels) * 100

    # Centre of attention (weighted centroid)
    y_coords, x_coords = np.mgrid[0:h, 0:w]
    total_weight = np.sum(heatmap) + 1e-8
    cx = np.sum(x_coords * heatmap) / total_weight
    cy = np.sum(y_coords * heatmap) / total_weight

    # Quadrant distribution
    mid_y, mid_x = h // 2, w // 2
    q_tl = np.mean(heatmap[:mid_y, :mid_x])
    q_tr = np.mean(heatmap[:mid_y, mid_x:])
    q_bl = np.mean(heatmap[mid_y:, :mid_x])
    q_br = np.mean(heatmap[mid_y:, mid_x:])

    quadrants = {'Top-Left': q_tl, 'Top-Right': q_tr,
                 'Bottom-Left': q_bl, 'Bottom-Right': q_br}
    dominant_quadrant = max(quadrants, key=quadrants.get)

    # Peak intensity
    peak_val = np.max(heatmap)
    mean_val = np.mean(heatmap)

    # Spread (standard deviation of active region)
    if active_pixels > 0:
        active_y = y_coords[heatmap >= threshold]
        active_x = x_coords[heatmap >= threshold]
        spread = np.sqrt(np.std(active_x)**2 + np.std(active_y)**2)
    else:
        spread = 0.0

    return {
        'focus_pct': focus_pct,
        'centre': (cx, cy),
        'dominant_quadrant': dominant_quadrant,
        'quadrants': quadrants,
        'peak_intensity': peak_val,
        'mean_intensity': mean_val,
        'spread': spread,
        'is_focused': focus_pct < 25,
        'is_diffuse': focus_pct > 50
    }


def run_gradcam(btn):
    with output_area:
        clear_output(wait=True)

        cls = class_dropdown.value
        img_name = image_dropdown.value
        img_path = os.path.join(TEST_DIR, cls, img_name)
        alpha = opacity_slider.value

        print('=' * 70)
        print('INTERACTIVE GRAD-CAM ANALYSIS')
        print('=' * 70)
        print(f'Image: {cls}/{img_name}')
        print(f'Overlay opacity: {alpha:.0%}')
        print('=' * 70)

        img_pil = Image.open(img_path).convert('RGB')

        # --- ResNet50 ---
        img_r = standard_transforms_resnet(img_pil).unsqueeze(0).to(DEVICE)
        resnet_trained.eval()
        with torch.no_grad():
            out_r = resnet_trained(img_r)
            probs_r = torch.softmax(out_r, dim=1)[0].cpu().numpy()
            pred_r = out_r.argmax(dim=1).item()

        cam_r = GradCAM(model=resnet_trained,
                         target_layers=[resnet_trained.base_model.layer4[-1]])
        heatmap_r = cam_r(input_tensor=img_r,
                           targets=[ClassifierOutputTarget(pred_r)])[0]

        # --- EfficientNetB0 ---
        img_e = standard_transforms_effnet(img_pil).unsqueeze(0).to(DEVICE)
        effnet_trained.eval()
        with torch.no_grad():
            out_e = effnet_trained(img_e)
            probs_e = torch.softmax(out_e, dim=1)[0].cpu().numpy()
            pred_e = out_e.argmax(dim=1).item()

        cam_e = GradCAM(model=effnet_trained,
                         target_layers=[effnet_trained.base_model.conv_head])
        heatmap_e = cam_e(input_tensor=img_e,
                           targets=[ClassifierOutputTarget(pred_e)])[0]

        # --- Attention Statistics ---
        stats_r = compute_attention_stats(heatmap_r)
        stats_e = compute_attention_stats(heatmap_e)

        # --- Main Visualisation ---
        img_resized = img_pil.resize((224, 224))
        img_np = np.array(img_resized).astype(np.float32) / 255.0

        overlay_r = show_cam_on_image(img_np, heatmap_r, use_rgb=True,
                                       image_weight=1 - alpha)
        overlay_e = show_cam_on_image(img_np, heatmap_e, use_rgb=True,
                                       image_weight=1 - alpha)

        fig, axes = plt.subplots(2, 4, figsize=(20, 10))

        # Row 1: ResNet50
        axes[0, 0].imshow(img_resized)
        axes[0, 0].set_title('Original MRI', fontsize=12, fontweight='bold')
        axes[0, 0].axis('off')

        axes[0, 1].imshow(heatmap_r, cmap='jet')
        axes[0, 1].set_title('ResNet50 Heatmap', fontsize=12, fontweight='bold')
        axes[0, 1].axis('off')

        axes[0, 2].imshow(overlay_r)
        correct_r = 'Correct' if CLASS_NAMES[pred_r] == cls else 'Wrong'
        axes[0, 2].set_title(
            f'ResNet50: {CLASS_NAMES[pred_r]} ({correct_r})\n'
            f'Confidence: {probs_r[pred_r]*100:.1f}%',
            fontsize=12, fontweight='bold',
            color='green' if correct_r == 'Correct' else 'red')
        axes[0, 2].axis('off')

        axes[0, 3].barh(CLASS_NAMES, probs_r * 100, color='steelblue')
        axes[0, 3].set_xlim(0, 100)
        axes[0, 3].set_title('ResNet50 Confidence', fontsize=12, fontweight='bold')
        axes[0, 3].set_xlabel('%')

        # Row 2: EfficientNetB0
        axes[1, 0].imshow(img_resized)
        axes[1, 0].set_title(f'True Label: {cls}', fontsize=12, fontweight='bold')
        axes[1, 0].axis('off')

        axes[1, 1].imshow(heatmap_e, cmap='jet')
        axes[1, 1].set_title('EfficientNetB0 Heatmap', fontsize=12, fontweight='bold')
        axes[1, 1].axis('off')

        axes[1, 2].imshow(overlay_e)
        correct_e = 'Correct' if CLASS_NAMES[pred_e] == cls else 'Wrong'
        axes[1, 2].set_title(
            f'EfficientNetB0: {CLASS_NAMES[pred_e]} ({correct_e})\n'
            f'Confidence: {probs_e[pred_e]*100:.1f}%',
            fontsize=12, fontweight='bold',
            color='green' if correct_e == 'Correct' else 'red')
        axes[1, 2].axis('off')

        axes[1, 3].barh(CLASS_NAMES, probs_e * 100, color='coral')
        axes[1, 3].set_xlim(0, 100)
        axes[1, 3].set_title('EfficientNetB0 Confidence', fontsize=12, fontweight='bold')
        axes[1, 3].set_xlabel('%')

        plt.suptitle(f'Interactive Grad-CAM Comparison: {cls}/{img_name}',
                     fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.show()

        # --- Attention Focus Comparison ---
        fig2, axes2 = plt.subplots(1, 2, figsize=(12, 5))

        # Quadrant heatmaps
        for ax, stats, name, cmap in [(axes2[0], stats_r, 'ResNet50', 'Blues'),
                                       (axes2[1], stats_e, 'EfficientNetB0', 'Oranges')]:
            q = stats['quadrants']
            quad_grid = np.array([[q['Top-Left'], q['Top-Right']],
                                  [q['Bottom-Left'], q['Bottom-Right']]])
            im = ax.imshow(quad_grid, cmap=cmap, vmin=0, vmax=max(q.values()) * 1.2)
            ax.set_xticks([0, 1])
            ax.set_xticklabels(['Left', 'Right'])
            ax.set_yticks([0, 1])
            ax.set_yticklabels(['Top', 'Bottom'])
            for i in range(2):
                for j in range(2):
                    ax.text(j, i, f'{quad_grid[i, j]:.3f}',
                            ha='center', va='center', fontsize=14, fontweight='bold')
            ax.set_title(f'{name}: Attention by Quadrant\n'
                         f'Focus: {stats["dominant_quadrant"]}',
                         fontsize=12, fontweight='bold')
            plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

        plt.suptitle('Where Each Model Concentrates Attention',
                     fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()

        # --- Detailed Report ---
        print('\n' + '=' * 70)
        print('HOW EACH MODEL SEES THIS IMAGE')
        print('=' * 70)

        for name, stats, probs, pred, correct in [
            ('ResNet50', stats_r, probs_r, pred_r, correct_r),
            ('EfficientNetB0', stats_e, probs_e, pred_e, correct_e)
        ]:
            print(f'\n--- {name} ---')
            print(f'  Prediction:        {CLASS_NAMES[pred]} ({probs[pred]*100:.2f}%) [{correct}]')
            print(f'  Peak attention:    {stats["peak_intensity"]:.4f}')
            print(f'  Mean attention:    {stats["mean_intensity"]:.4f}')
            print(f'  Focus area:        {stats["focus_pct"]:.1f}% of image')
            print(f'  Attention centre:  ({stats["centre"][0]:.0f}, {stats["centre"][1]:.0f}) pixels')
            print(f'  Dominant region:   {stats["dominant_quadrant"]}')
            print(f'  Spatial spread:    {stats["spread"]:.1f} pixels')

            if stats['is_focused']:
                print(f'  Pattern:           FOCUSED -- the model locks onto a small, '
                      f'specific region, suggesting a clear discriminative feature.')
            elif stats['is_diffuse']:
                print(f'  Pattern:           DIFFUSE -- the model looks at a broad area, '
                      f'which may indicate reliance on global texture or shape.')
            else:
                print(f'  Pattern:           MODERATE -- balanced attention across '
                      f'a mid-sized region.')

        # --- Agreement Analysis ---
        print('\n' + '-' * 70)
        print('MODEL AGREEMENT ANALYSIS')
        print('-' * 70)

        agree = (pred_r == pred_e)
        print(f'  Both models agree:  {"Yes" if agree else "No"}')

        if agree:
            conf_diff = abs(probs_r[pred_r] - probs_e[pred_e]) * 100
            print(f'  Confidence gap:     {conf_diff:.1f}%')
            if conf_diff < 5:
                print(f'  Interpretation:     Both models are equally confident, '
                      f'indicating a clear case.')
            else:
                more_conf = 'ResNet50' if probs_r[pred_r] > probs_e[pred_e] else 'EfficientNetB0'
                print(f'  Interpretation:     {more_conf} is more confident. '
                      f'Different architectures may extract different features.')
        else:
            print(f'  ResNet50 says:      {CLASS_NAMES[pred_r]} ({probs_r[pred_r]*100:.1f}%)')
            print(f'  EfficientNetB0 says: {CLASS_NAMES[pred_e]} ({probs_e[pred_e]*100:.1f}%)')
            print(f'  Interpretation:     The models disagree. Compare the heatmaps above '
                  f'to see which regions each model uses. Disagreements often occur at '
                  f'class boundaries (e.g., glioma vs meningioma).')

        # --- Spatial overlap ---
        binary_r = (heatmap_r >= 0.5).astype(float)
        binary_e = (heatmap_e >= 0.5).astype(float)
        intersection = np.sum(binary_r * binary_e)
        union = np.sum(np.clip(binary_r + binary_e, 0, 1))
        iou = intersection / (union + 1e-8)

        print(f'\n  Attention overlap (IoU): {iou:.2%}')
        if iou > 0.5:
            print(f'  Both models focus on similar regions, suggesting they '
                  f'detect the same anatomical features.')
        elif iou > 0.2:
            print(f'  Partial overlap -- the models share some attention but '
                  f'also use different cues.')
        else:
            print(f'  Low overlap -- each architecture learns distinct visual '
                  f'patterns for classification.')

        print('=' * 70)


run_btn = widgets.Button(description='Run Grad-CAM', button_style='info',
                          layout=widgets.Layout(width='140px'))
run_btn.on_click(run_gradcam)

display(widgets.HBox([class_dropdown, image_dropdown, opacity_slider, run_btn]))
display(output_area)

##  Interactive SHAP

In [ ]:
# Interactive SHAP: Select an image and compare both models

import ipywidgets as widgets
from IPython.display import display, clear_output

if 'test_image_paths' not in dir():
    test_image_paths = {}
    for cls in sorted(os.listdir(TEST_DIR)):
        cls_path = os.path.join(TEST_DIR, cls)
        if os.path.isdir(cls_path):
            imgs = sorted([f for f in os.listdir(cls_path)
                           if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
            test_image_paths[cls] = imgs

shap_class_dd = widgets.Dropdown(
    options=list(test_image_paths.keys()),
    description='Class:',
    style={'description_width': '60px'}
)

shap_image_dd = widgets.Dropdown(
    options=test_image_paths[shap_class_dd.value][:20],
    description='Image:',
    style={'description_width': '60px'}
)

shap_evals_slider = widgets.IntSlider(
    value=100, min=20, max=200, step=10,
    description='max_evals:',
    style={'description_width': '70px'}
)

shap_output = widgets.Output()

def shap_update_images(*args):
    shap_image_dd.options = test_image_paths[shap_class_dd.value][:20]

shap_class_dd.observe(shap_update_images, names='value')

mean_arr = np.array([0.485, 0.456, 0.406])
std_arr = np.array([0.229, 0.224, 0.225])

def make_predict_fn(model):
    def predict_fn(images):
        preds = []
        for img in images:
            img_float = img.astype(np.float32) / 255.0
            img_norm = (img_float - mean_arr) / std_arr
            img_chw = np.transpose(img_norm, (2, 0, 1))
            tensor = torch.from_numpy(img_chw).float().unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                output = model(tensor)
                probs = torch.nn.functional.softmax(output, dim=1)
                preds.append(probs.cpu().numpy()[0])
        return np.array(preds)
    return predict_fn


def compute_shap_stats(shap_values_single, class_names, pred_class):
    """Compute detailed statistics from SHAP values for one image."""
    n_classes = len(class_names)
    stats = {}

    for j in range(n_classes):
        sv = shap_values_single.values[..., j]
        sv_abs = np.sum(np.abs(sv), axis=-1) if sv.ndim == 3 else np.abs(sv)
        sv_signed = np.sum(sv, axis=-1) if sv.ndim == 3 else sv

        total_pos = np.sum(sv_signed[sv_signed > 0])
        total_neg = np.sum(sv_signed[sv_signed < 0])
        total_importance = np.sum(sv_abs)

        # Top 10% most important pixels
        threshold = np.percentile(sv_abs, 90)
        top_mask = sv_abs >= threshold
        top_pct = np.sum(top_mask) / sv_abs.size * 100

        # Centre of importance
        h, w = sv_abs.shape
        y_coords, x_coords = np.mgrid[0:h, 0:w]
        total_w = np.sum(sv_abs) + 1e-8
        cx = np.sum(x_coords * sv_abs) / total_w
        cy = np.sum(y_coords * sv_abs) / total_w

        stats[class_names[j]] = {
            'total_importance': total_importance,
            'positive_support': total_pos,
            'negative_opposition': abs(total_neg),
            'net_effect': total_pos + total_neg,
            'top_region_pct': top_pct,
            'importance_centre': (cx, cy),
            'peak_value': np.max(sv_abs),
            'is_predicted': j == pred_class
        }

    return stats


def run_shap(btn):
    with shap_output:
        clear_output(wait=True)

        cls = shap_class_dd.value
        img_name = shap_image_dd.value
        img_path = os.path.join(TEST_DIR, cls, img_name)
        max_evals = shap_evals_slider.value

        print('=' * 70)
        print('INTERACTIVE SHAP ANALYSIS')
        print('=' * 70)
        print(f'Image:     {cls}/{img_name}')
        print(f'max_evals: {max_evals}')
        print(f'Estimated time: 2-5 minutes per model')
        print('=' * 70)

        img_pil = Image.open(img_path).convert('RGB').resize((224, 224))
        img_uint8 = np.array(img_pil).astype(np.uint8)
        images_array = np.expand_dims(img_uint8, axis=0)

        masker = shap.maskers.Image("inpaint_telea", img_uint8.shape)

        all_results = {}
        all_stats = {}

        for name, model in [('ResNet50', resnet_trained),
                             ('EfficientNetB0', effnet_trained)]:
            print(f'\n--- {name} ---')
            model.eval()
            predict_fn = make_predict_fn(model)

            pred_probs = predict_fn(images_array)[0]
            pred_class = np.argmax(pred_probs)
            print(f'Prediction: {CLASS_NAMES[pred_class]} '
                  f'({pred_probs[pred_class]*100:.1f}%)')

            explainer = shap.Explainer(predict_fn, masker,
                                        output_names=CLASS_NAMES)
            print(f'Computing SHAP values...')
            shap_values = explainer(images_array, max_evals=max_evals,
                                     batch_size=10)

            all_results[name] = {
                'shap_values': shap_values,
                'pred_class': pred_class,
                'pred_probs': pred_probs
            }

            all_stats[name] = compute_shap_stats(
                shap_values[0], CLASS_NAMES, pred_class)

            print(f'{name} SHAP complete.')
            torch.cuda.empty_cache()
            gc.collect()

        # --- Main Visualisation: Per-Class SHAP Maps ---
        n_classes = len(CLASS_NAMES)
        fig, axes = plt.subplots(2, n_classes + 1,
                                  figsize=(5 * (n_classes + 1), 10))

        for row, (name, res) in enumerate(all_results.items()):
            axes[row, 0].imshow(img_uint8)
            correct = 'Correct' if CLASS_NAMES[res['pred_class']] == cls \
                      else 'Wrong'
            axes[row, 0].set_title(
                f'{name}\nPred: {CLASS_NAMES[res["pred_class"]]} ({correct})\n'
                f'Conf: {res["pred_probs"][res["pred_class"]]*100:.1f}%',
                fontsize=11, fontweight='bold',
                color='green' if correct == 'Correct' else 'red')
            axes[row, 0].axis('off')

            for j, c in enumerate(CLASS_NAMES):
                sv = res['shap_values'][0].values[..., j]
                sv_abs = np.sum(np.abs(sv), axis=-1) if sv.ndim == 3 \
                         else np.abs(sv)
                axes[row, j + 1].imshow(img_uint8, alpha=0.3)
                axes[row, j + 1].imshow(sv_abs, cmap='hot', alpha=0.7)
                marker = ' *' if j == res['pred_class'] else ''
                axes[row, j + 1].set_title(f'{c}{marker}',
                                            fontsize=11, fontweight='bold')
                axes[row, j + 1].axis('off')

        plt.suptitle(
            f'SHAP Feature Importance: {cls}/{img_name}\n'
            f'(* = predicted class, max_evals={max_evals})',
            fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()

        # --- Importance Bar Chart ---
        fig2, axes2 = plt.subplots(1, 2, figsize=(14, 5))

        for ax, (name, stats) in zip(axes2, all_stats.items()):
            classes_sorted = sorted(stats.keys(),
                                     key=lambda c: stats[c]['total_importance'],
                                     reverse=True)
            importances = [stats[c]['total_importance'] for c in classes_sorted]
            colours = ['#e74c3c' if stats[c]['is_predicted'] else '#3498db'
                       for c in classes_sorted]

            ax.barh(classes_sorted, importances, color=colours, edgecolor='black')
            ax.set_xlabel('Total SHAP Importance')
            ax.set_title(f'{name}: Feature Importance per Class',
                         fontsize=12, fontweight='bold')
            for i, (c, v) in enumerate(zip(classes_sorted, importances)):
                ax.text(v + max(importances) * 0.01, i, f'{v:.3f}',
                        va='center', fontsize=10)

        plt.suptitle('Which Classes Does Each Model Find Evidence For?',
                     fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()

        # --- Positive vs Negative Contributions ---
        fig3, axes3 = plt.subplots(1, 2, figsize=(14, 5))

        for ax, (name, stats) in zip(axes3, all_stats.items()):
            pos_vals = [stats[c]['positive_support'] for c in CLASS_NAMES]
            neg_vals = [-stats[c]['negative_opposition'] for c in CLASS_NAMES]

            x = np.arange(len(CLASS_NAMES))
            ax.bar(x, pos_vals, 0.6, label='Supporting (positive)',
                   color='#2ecc71', edgecolor='black')
            ax.bar(x, neg_vals, 0.6, label='Opposing (negative)',
                   color='#e74c3c', edgecolor='black')
            ax.set_xticks(x)
            ax.set_xticklabels(CLASS_NAMES, rotation=15)
            ax.set_ylabel('SHAP Value Sum')
            ax.set_title(f'{name}: Support vs Opposition',
                         fontsize=12, fontweight='bold')
            ax.legend(fontsize=9)
            ax.axhline(y=0, color='black', linewidth=0.8)

        plt.suptitle('How Image Regions Support or Oppose Each Class',
                     fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()

        # --- Detailed Report ---
        print('\n' + '=' * 70)
        print('HOW EACH MODEL INTERPRETS THIS IMAGE')
        print('=' * 70)

        for name, stats in all_stats.items():
            res = all_results[name]
            pred = res['pred_class']
            correct = 'Correct' if CLASS_NAMES[pred] == cls else 'Wrong'

            print(f'\n{"=" * 35}')
            print(f'{name}')
            print(f'{"=" * 35}')
            print(f'  Prediction: {CLASS_NAMES[pred]} '
                  f'({res["pred_probs"][pred]*100:.2f}%) [{correct}]')

            print(f'\n  Per-Class SHAP Breakdown:')
            print(f'  {"Class":<15s} {"Support":>10s} {"Oppose":>10s} '
                  f'{"Net":>10s} {"Peak":>8s}')
            print(f'  {"-" * 55}')

            for c in CLASS_NAMES:
                s = stats[c]
                marker = '  <-- predicted' if s['is_predicted'] else ''
                print(f'  {c:<15s} {s["positive_support"]:>10.4f} '
                      f'{s["negative_opposition"]:>10.4f} '
                      f'{s["net_effect"]:>+10.4f} '
                      f'{s["peak_value"]:>8.4f}{marker}')

            # Interpretation
            pred_stats = stats[CLASS_NAMES[pred]]
            print(f'\n  Interpretation:')
            print(f'  The model found {pred_stats["positive_support"]:.4f} '
                  f'total supporting evidence for {CLASS_NAMES[pred]}.')

            # Check if second-highest class is close
            sorted_classes = sorted(stats.keys(),
                                     key=lambda c: stats[c]['net_effect'],
                                     reverse=True)
            if len(sorted_classes) >= 2:
                top = stats[sorted_classes[0]]['net_effect']
                second = stats[sorted_classes[1]]['net_effect']
                if top > 0 and second > 0:
                    ratio = second / (top + 1e-8)
                    if ratio > 0.7:
                        print(f'  The evidence for {sorted_classes[1]} is also '
                              f'strong ({ratio:.0%} of the top class), '
                              f'suggesting visual similarity between the two.')
                    else:
                        print(f'  The evidence strongly favours '
                              f'{sorted_classes[0]} over other classes.')

            cx, cy = pred_stats['importance_centre']
            print(f'  Key features are centred around pixel '
                  f'({cx:.0f}, {cy:.0f}) in the 224x224 image.')

        # --- Agreement Analysis ---
        print('\n' + '-' * 70)
        print('MODEL COMPARISON')
        print('-' * 70)

        r_pred = all_results['ResNet50']['pred_class']
        e_pred = all_results['EfficientNetB0']['pred_class']
        agree = (r_pred == e_pred)

        print(f'  Both models agree: {"Yes" if agree else "No"}')

        if agree:
            # Compare which model has stronger evidence
            r_net = all_stats['ResNet50'][CLASS_NAMES[r_pred]]['net_effect']
            e_net = all_stats['EfficientNetB0'][CLASS_NAMES[e_pred]]['net_effect']
            stronger = 'ResNet50' if r_net > e_net else 'EfficientNetB0'
            print(f'  {stronger} shows stronger SHAP evidence for the '
                  f'predicted class.')

            # Compare spatial focus
            r_cx, r_cy = all_stats['ResNet50'][CLASS_NAMES[r_pred]]['importance_centre']
            e_cx, e_cy = all_stats['EfficientNetB0'][CLASS_NAMES[e_pred]]['importance_centre']
            dist = np.sqrt((r_cx - e_cx)**2 + (r_cy - e_cy)**2)
            print(f'  Feature centre distance: {dist:.1f} pixels')
            if dist < 30:
                print(f'  Both models focus on the same region of the scan.')
            else:
                print(f'  The models use different spatial regions, '
                      f'suggesting they learn complementary features.')
        else:
            print(f'  ResNet50:       {CLASS_NAMES[r_pred]}')
            print(f'  EfficientNetB0: {CLASS_NAMES[e_pred]}')
            print(f'  The models disagree. Examine the SHAP maps above to '
                  f'understand which image features drive each prediction.')

        print('=' * 70)


shap_btn = widgets.Button(description='Run SHAP', button_style='warning',
                           layout=widgets.Layout(width='120px'))
shap_btn.on_click(run_shap)

display(widgets.HBox([shap_class_dd, shap_image_dd, shap_evals_slider,
                       shap_btn]))
display(shap_output)

## Combined XAI Analysis

In [ ]:
# Combined XAI Analysis: Grad-CAM + SHAP Working Together
# This cell runs both methods on the same image and analyses
# whether they agree on which regions drive the prediction.

import ipywidgets as widgets
from IPython.display import display, clear_output
from scipy.ndimage import gaussian_filter

# Collect test images
test_image_paths = {}
for cls in sorted(os.listdir(TEST_DIR)):
    cls_path = os.path.join(TEST_DIR, cls)
    if os.path.isdir(cls_path):
        imgs = sorted([f for f in os.listdir(cls_path)
                       if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        test_image_paths[cls] = imgs

class_dd = widgets.Dropdown(
    options=list(test_image_paths.keys()),
    description='Class:',
    style={'description_width': '60px'}
)
image_dd = widgets.Dropdown(
    options=test_image_paths[class_dd.value][:20],
    description='Image:',
    style={'description_width': '60px'}
)
evals_slider = widgets.IntSlider(
    value=100, min=20, max=200, step=10,
    description='max_evals:',
    style={'description_width': '70px'}
)
combined_output = widgets.Output()

def update_imgs(*args):
    image_dd.options = test_image_paths[class_dd.value][:20]
class_dd.observe(update_imgs, names='value')

mean_arr = np.array([0.485, 0.456, 0.406])
std_arr = np.array([0.229, 0.224, 0.225])

def make_predict_fn(model):
    def predict_fn(images):
        preds = []
        for img in images:
            img_float = img.astype(np.float32) / 255.0
            img_norm = (img_float - mean_arr) / std_arr
            img_chw = np.transpose(img_norm, (2, 0, 1))
            tensor = torch.from_numpy(img_chw).float().unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                output = model(tensor)
                probs = torch.nn.functional.softmax(output, dim=1)
                preds.append(probs.cpu().numpy()[0])
        return np.array(preds)
    return predict_fn


def run_combined(btn):
    with combined_output:
        clear_output(wait=True)

        cls = class_dd.value
        img_name = image_dd.value
        img_path = os.path.join(TEST_DIR, cls, img_name)
        max_evals = evals_slider.value

        print('=' * 70)
        print('COMBINED XAI ANALYSIS: GRAD-CAM + SHAP')
        print('=' * 70)
        print(f'Image:     {cls}/{img_name}')
        print(f'SHAP evals: {max_evals}')
        print('Running both methods on both models...')
        print('=' * 70)

        img_pil = Image.open(img_path).convert('RGB')
        img_resized = img_pil.resize((224, 224))
        img_np = np.array(img_resized).astype(np.float32) / 255.0
        img_uint8 = np.array(img_resized).astype(np.uint8)
        images_array = np.expand_dims(img_uint8, axis=0)

        masker = shap.maskers.Image("inpaint_telea", img_uint8.shape)

        models_info = {
            'ResNet50': {
                'model': resnet_trained,
                'transforms': standard_transforms_resnet,
                'gradcam_layer': resnet_trained.base_model.layer4[-1],
            },
            'EfficientNetB0': {
                'model': effnet_trained,
                'transforms': standard_transforms_effnet,
                'gradcam_layer': effnet_trained.base_model.conv_head,
            }
        }

        results = {}

        for name, info in models_info.items():
            model = info['model']
            model.eval()

            # Prediction
            img_t = info['transforms'](img_pil).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                out = model(img_t)
                probs = torch.softmax(out, dim=1)[0].cpu().numpy()
                pred = out.argmax(dim=1).item()

            # Grad-CAM
            print(f'\n[{name}] Running Grad-CAM...')
            cam = GradCAM(model=model,
                           target_layers=[info['gradcam_layer']])
            gradcam_map = cam(input_tensor=img_t,
                               targets=[ClassifierOutputTarget(pred)])[0]

            # SHAP
            print(f'[{name}] Running SHAP (max_evals={max_evals})...')
            predict_fn = make_predict_fn(model)
            explainer = shap.Explainer(predict_fn, masker,
                                        output_names=CLASS_NAMES)
            shap_values = explainer(images_array, max_evals=max_evals,
                                     batch_size=10)

            # SHAP importance for predicted class
            sv_pred = shap_values[0].values[..., pred]
            shap_map = np.sum(np.abs(sv_pred), axis=-1) if sv_pred.ndim == 3 \
                       else np.abs(sv_pred)

            # Normalise both maps to [0, 1]
            gc_norm = (gradcam_map - gradcam_map.min()) / \
                      (gradcam_map.max() - gradcam_map.min() + 1e-8)
            sh_norm = (shap_map - shap_map.min()) / \
                      (shap_map.max() - shap_map.min() + 1e-8)

            # Smooth SHAP to match Grad-CAM resolution
            sh_smooth = gaussian_filter(sh_norm, sigma=3)
            sh_smooth = (sh_smooth - sh_smooth.min()) / \
                        (sh_smooth.max() - sh_smooth.min() + 1e-8)

            # Combined Agreement Map
            # Multiply: high only where BOTH methods agree
            agreement_map = gc_norm * sh_smooth

            # Disagreement Maps
            gradcam_only = np.clip(gc_norm - sh_smooth, 0, 1)
            shap_only = np.clip(sh_smooth - gc_norm, 0, 1)

            # Statistics
            gc_binary = (gc_norm >= 0.5).astype(float)
            sh_binary = (sh_smooth >= 0.5).astype(float)
            intersection = np.sum(gc_binary * sh_binary)
            union = np.sum(np.clip(gc_binary + sh_binary, 0, 1))
            iou = intersection / (union + 1e-8)

            # Correlation
            corr = np.corrcoef(gc_norm.flatten(), sh_smooth.flatten())[0, 1]

            # Signed SHAP for support/oppose
            sv_signed = np.sum(sv_pred, axis=-1) if sv_pred.ndim == 3 \
                        else sv_pred
            support = np.sum(sv_signed[sv_signed > 0])
            oppose = np.sum(np.abs(sv_signed[sv_signed < 0]))

            results[name] = {
                'pred': pred, 'probs': probs,
                'gradcam_map': gc_norm, 'shap_map': sh_smooth,
                'agreement_map': agreement_map,
                'gradcam_only': gradcam_only, 'shap_only': shap_only,
                'iou': iou, 'correlation': corr,
                'support': support, 'oppose': oppose,
                'shap_values': shap_values
            }

            print(f'[{name}] Complete. IoU={iou:.3f}, Correlation={corr:.3f}')
            torch.cuda.empty_cache()
            gc.collect()


        # FIGURE 1: Side-by-side for each model (6 panels per row)

        fig1, axes1 = plt.subplots(2, 6, figsize=(30, 10))

        for row, (name, res) in enumerate(results.items()):
            pred = res['pred']
            correct = 'Correct' if CLASS_NAMES[pred] == cls else 'Wrong'
            conf = res['probs'][pred] * 100

            # (a) Original
            axes1[row, 0].imshow(img_resized)
            axes1[row, 0].set_title(f'{name}\nTrue: {cls}',
                                     fontsize=11, fontweight='bold')
            axes1[row, 0].axis('off')

            # (b) Grad-CAM
            axes1[row, 1].imshow(img_np)
            axes1[row, 1].imshow(res['gradcam_map'], cmap='jet', alpha=0.6)
            axes1[row, 1].set_title('Grad-CAM\n(Where it looks)',
                                     fontsize=11, fontweight='bold')
            axes1[row, 1].axis('off')

            # (c) SHAP
            axes1[row, 2].imshow(img_np)
            axes1[row, 2].imshow(res['shap_map'], cmap='hot', alpha=0.6)
            axes1[row, 2].set_title('SHAP\n(How much each part matters)',
                                     fontsize=11, fontweight='bold')
            axes1[row, 2].axis('off')

            # (d) Agreement (both high)
            axes1[row, 3].imshow(img_np)
            axes1[row, 3].imshow(res['agreement_map'], cmap='Greens',
                                  alpha=0.7)
            axes1[row, 3].set_title('Agreement\n(Both methods agree)',
                                     fontsize=11, fontweight='bold',
                                     color='green')
            axes1[row, 3].axis('off')

            # (e) Grad-CAM only (attention without importance)
            axes1[row, 4].imshow(img_np)
            axes1[row, 4].imshow(res['gradcam_only'], cmap='Blues', alpha=0.7)
            axes1[row, 4].set_title('Grad-CAM Only\n(Looks but not important)',
                                     fontsize=11, fontweight='bold',
                                     color='blue')
            axes1[row, 4].axis('off')

            # (f) SHAP only (importance without attention)
            axes1[row, 5].imshow(img_np)
            axes1[row, 5].imshow(res['shap_only'], cmap='Oranges', alpha=0.7)
            axes1[row, 5].set_title('SHAP Only\n(Important but not attended)',
                                     fontsize=11, fontweight='bold',
                                     color='darkorange')
            axes1[row, 5].axis('off')

        plt.suptitle(
            f'Combined XAI: How Grad-CAM and SHAP Work Together\n'
            f'Image: {cls}/{img_name}',
            fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.show()


        # FIGURE 2: Agreement Metrics Comparison

        fig2, axes2 = plt.subplots(1, 3, figsize=(18, 5))

        # (a) IoU and Correlation bars
        model_names = list(results.keys())
        ious = [results[n]['iou'] for n in model_names]
        corrs = [results[n]['correlation'] for n in model_names]

        x = np.arange(len(model_names))
        width = 0.35
        axes2[0].bar(x - width/2, ious, width, label='IoU (spatial overlap)',
                     color='#2ecc71', edgecolor='black')
        axes2[0].bar(x + width/2, corrs, width,
                     label='Correlation (intensity)',
                     color='#3498db', edgecolor='black')
        axes2[0].set_xticks(x)
        axes2[0].set_xticklabels(model_names)
        axes2[0].set_ylim(0, 1.1)
        axes2[0].set_ylabel('Score')
        axes2[0].set_title('Grad-CAM vs SHAP Agreement',
                           fontsize=12, fontweight='bold')
        axes2[0].legend()
        for i, (iou_v, corr_v) in enumerate(zip(ious, corrs)):
            axes2[0].text(i - width/2, iou_v + 0.02, f'{iou_v:.3f}',
                         ha='center', fontsize=10, fontweight='bold')
            axes2[0].text(i + width/2, corr_v + 0.02, f'{corr_v:.3f}',
                         ha='center', fontsize=10, fontweight='bold')

        # (b) SHAP Support vs Oppose
        supports = [results[n]['support'] for n in model_names]
        opposes = [-results[n]['oppose'] for n in model_names]

        axes2[1].bar(model_names, supports, color='#2ecc71',
                     edgecolor='black', label='Supporting evidence')
        axes2[1].bar(model_names, opposes, color='#e74c3c',
                     edgecolor='black', label='Opposing evidence')
        axes2[1].axhline(y=0, color='black', linewidth=0.8)
        axes2[1].set_ylabel('SHAP Value Sum')
        axes2[1].set_title(f'Evidence For/Against: {CLASS_NAMES[results[model_names[0]]["pred"]]}',
                           fontsize=12, fontweight='bold')
        axes2[1].legend()

        # (c) Confidence comparison
        for name in model_names:
            axes2[2].barh(CLASS_NAMES, results[name]['probs'] * 100,
                         alpha=0.6, label=name, edgecolor='black')
        axes2[2].set_xlabel('Confidence (%)')
        axes2[2].set_title('Prediction Confidence',
                           fontsize=12, fontweight='bold')
        axes2[2].legend()
        axes2[2].set_xlim(0, 100)

        plt.suptitle('Quantitative XAI Comparison',
                     fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()

        # FIGURE 3: Pixel-level scatter (Grad-CAM vs SHAP)

        fig3, axes3 = plt.subplots(1, 2, figsize=(14, 6))

        for ax, (name, res) in zip(axes3, results.items()):
            gc_flat = res['gradcam_map'].flatten()
            sh_flat = res['shap_map'].flatten()

            # Subsample for speed
            idx = np.random.choice(len(gc_flat), min(2000, len(gc_flat)),
                                    replace=False)

            ax.scatter(gc_flat[idx], sh_flat[idx], alpha=0.15, s=8,
                      color='#8e44ad')
            ax.plot([0, 1], [0, 1], 'r--', linewidth=1.5, label='Perfect agreement')
            ax.set_xlabel('Grad-CAM Intensity')
            ax.set_ylabel('SHAP Importance')
            ax.set_title(f'{name}\nr = {res["correlation"]:.3f}',
                         fontsize=12, fontweight='bold')
            ax.legend()
            ax.set_xlim(-0.05, 1.05)
            ax.set_ylim(-0.05, 1.05)
            ax.set_aspect('equal')
            ax.grid(True, alpha=0.2)

        plt.suptitle(
            'Pixel-Level Correlation: Does Grad-CAM Attention\n'
            'Match SHAP Importance?',
            fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()

        # TEXT REPORT

        print('\n' + '=' * 70)
        print('COMBINED XAI REPORT: HOW THE METHODS WORK TOGETHER')
        print('=' * 70)

        for name, res in results.items():
            pred = res['pred']
            correct = 'Correct' if CLASS_NAMES[pred] == cls else 'Wrong'

            print(f'\n{"=" * 35}')
            print(f'{name}')
            print(f'{"=" * 35}')
            print(f'  Prediction: {CLASS_NAMES[pred]} '
                  f'({res["probs"][pred]*100:.2f}%) [{correct}]')

            print(f'\n  1. Grad-CAM (spatial attention):')
            gc_focus = np.sum(res['gradcam_map'] >= 0.5) / \
                       res['gradcam_map'].size * 100
            print(f'     Focus area: {gc_focus:.1f}% of the image')
            if gc_focus < 20:
                print(f'     The model has a tight, localised focus, '
                      f'typically on a specific anatomical feature.')
            elif gc_focus < 40:
                print(f'     The model examines a moderate region, '
                      f'looking at both the lesion and surrounding context.')
            else:
                print(f'     The model uses a broad, diffuse attention '
                      f'pattern across a large portion of the scan.')

            print(f'\n  2. SHAP (feature importance):')
            print(f'     Supporting evidence:  {res["support"]:.4f}')
            print(f'     Opposing evidence:    {res["oppose"]:.4f}')
            ratio = res['support'] / (res['support'] + res['oppose'] + 1e-8)
            print(f'     Support ratio:        {ratio:.1%}')
            if ratio > 0.7:
                print(f'     Strong evidence -- the image features clearly '
                      f'point to {CLASS_NAMES[pred]}.')
            elif ratio > 0.5:
                print(f'     Mixed evidence -- some regions support the '
                      f'prediction while others suggest alternatives.')
            else:
                print(f'     Weak evidence -- the model is uncertain, '
                      f'opposing features nearly outweigh support.')

            print(f'\n  3. Agreement between methods:')
            print(f'     Spatial overlap (IoU):     {res["iou"]:.3f}')
            print(f'     Intensity correlation (r): {res["correlation"]:.3f}')

            if res['iou'] > 0.4 and res['correlation'] > 0.5:
                print(f'\n     STRONG AGREEMENT:')
                print(f'     Both Grad-CAM and SHAP highlight the same '
                      f'regions. This means the area the model attends to')
                print(f'     (Grad-CAM) is also where the most '
                      f'decision-critical features exist (SHAP).')
                print(f'     This is the ideal outcome: the model focuses '
                      f'on clinically relevant anatomy.')
            elif res['iou'] > 0.2 or res['correlation'] > 0.3:
                print(f'\n     PARTIAL AGREEMENT:')
                print(f'     There is some overlap, but each method also '
                      f'identifies unique regions.')
                print(f'     Grad-CAM-only regions (blue in figure): '
                      f'the model activates here but SHAP shows')
                print(f'     these areas do not strongly influence the '
                      f'final probability.')
                print(f'     SHAP-only regions (orange in figure): '
                      f'these areas change the prediction when masked')
                print(f'     but do not produce strong feature map '
                      f'activations in the CNN layers.')
            else:
                print(f'\n     LOW AGREEMENT:')
                print(f'     The methods highlight different regions. '
                      f'This may indicate:')
                print(f'     - The model uses low-level texture features '
                      f'(captured by SHAP) rather than')
                print(f'       high-level spatial features '
                      f'(captured by Grad-CAM).')
                print(f'     - Grad-CAM resolution (7x7 for ResNet50) '
                      f'is too coarse to capture fine patterns.')
                print(f'     - The decision boundary depends on subtle '
                      f'pixel-level differences.')

        # Cross-model comparison
        print('\n' + '=' * 70)
        print('CROSS-MODEL ANALYSIS')
        print('=' * 70)

        r_res = results['ResNet50']
        e_res = results['EfficientNetB0']

        r_pred = r_res['pred']
        e_pred = e_res['pred']

        if r_pred == e_pred:
            print(f'\n  Both models predict: {CLASS_NAMES[r_pred]}')
            print(f'  ResNet50 confidence:       {r_res["probs"][r_pred]*100:.2f}%')
            print(f'  EfficientNetB0 confidence: {e_res["probs"][e_pred]*100:.2f}%')

            # Compare agreement levels
            if r_res['iou'] > e_res['iou']:
                better = 'ResNet50'
            else:
                better = 'EfficientNetB0'
            print(f'\n  {better} shows higher Grad-CAM/SHAP agreement '
                  f'(IoU: {max(r_res["iou"], e_res["iou"]):.3f}),')
            print(f'  meaning its attention aligns better with its '
                  f'feature importance.')

            # Cross-model Grad-CAM overlap
            gc_cross_inter = np.sum(
                (r_res['gradcam_map'] >= 0.5).astype(float) *
                (e_res['gradcam_map'] >= 0.5).astype(float))
            gc_cross_union = np.sum(np.clip(
                (r_res['gradcam_map'] >= 0.5).astype(float) +
                (e_res['gradcam_map'] >= 0.5).astype(float), 0, 1))
            gc_cross_iou = gc_cross_inter / (gc_cross_union + 1e-8)

            print(f'\n  Cross-model Grad-CAM overlap: {gc_cross_iou:.3f}')
            if gc_cross_iou > 0.4:
                print(f'  Both architectures attend to similar regions, '
                      f'suggesting the discriminative')
                print(f'  features are architecture-independent '
                      f'(likely genuine anatomical markers).')
            else:
                print(f'  Each architecture focuses on different regions, '
                      f'suggesting they learn')
                print(f'  complementary feature representations. '
                      f'This supports the value of using')
                print(f'  multiple architectures for robust diagnosis.')
        else:
            print(f'\n  Models DISAGREE:')
            print(f'  ResNet50:       {CLASS_NAMES[r_pred]} '
                  f'({r_res["probs"][r_pred]*100:.1f}%)')
            print(f'  EfficientNetB0: {CLASS_NAMES[e_pred]} '
                  f'({e_res["probs"][e_pred]*100:.1f}%)')
            print(f'\n  Compare the agreement maps above to see what each '
                  f'model considers decisive.')
            print(f'  Disagreements often occur with glioma vs '
                  f'meningioma due to overlapping visual features.')

        # Clinical Relevance
        print('\n' + '=' * 70)
        print('CLINICAL RELEVANCE')
        print('=' * 70)

        both_correct = (CLASS_NAMES[r_pred] == cls and
                        CLASS_NAMES[e_pred] == cls)
        any_correct = (CLASS_NAMES[r_pred] == cls or
                       CLASS_NAMES[e_pred] == cls)

        if both_correct:
            avg_iou = (r_res['iou'] + e_res['iou']) / 2
            avg_corr = (r_res['correlation'] + e_res['correlation']) / 2
            print(f'\n  Both models correctly classify this scan.')
            if avg_iou > 0.3 and avg_corr > 0.4:
                print(f'  The XAI methods confirm the models use '
                      f'consistent, overlapping features.')
                print(f'  A clinician reviewing these heatmaps would see '
                      f'the model focusing on the')
                print(f'  expected tumour region, supporting trust in '
                      f'the automated classification.')
            else:
                print(f'  While predictions are correct, the explanation '
                      f'methods show moderate')
                print(f'  disagreement about which exact features drive '
                      f'the decision.')
                print(f'  This suggests the model may rely on a '
                      f'combination of features that')
                print(f'  individual XAI methods capture differently.')
        elif any_correct:
            correct_model = 'ResNet50' if CLASS_NAMES[r_pred] == cls \
                            else 'EfficientNetB0'
            wrong_model = 'EfficientNetB0' if correct_model == 'ResNet50' \
                          else 'ResNet50'
            print(f'\n  {correct_model} is correct, {wrong_model} is wrong.')
            print(f'  Comparing their XAI outputs reveals why:')
            print(f'  the correct model likely attends to the actual '
                  f'tumour region while the')
            print(f'  incorrect model may focus on irrelevant anatomy '
                  f'or imaging artefacts.')
        else:
            print(f'\n  Both models misclassify this scan.')
            print(f'  The XAI analysis is especially valuable here -- '
                  f'check the heatmaps')
            print(f'  to identify what confuses both architectures.')

        print('\n' + '=' * 70)
        print('ANALYSIS COMPLETE')
        print('=' * 70)


run_btn = widgets.Button(description='Run Combined XAI',
                          button_style='success',
                          layout=widgets.Layout(width='160px'))
run_btn.on_click(run_combined)

display(widgets.HBox([class_dd, image_dd, evals_slider, run_btn]))
display(combined_output)